# 3. Preprocessing & Ekstraksi Fitur — Kecamatan Kedungpring, Kabupaten Lamongan

**Mata Kuliah:** Proyek Sains Data — Semester 5
**Wilayah:** Kecamatan Kedungpring, Kabupaten Lamongan
**Polutan:** NO₂, CO, SO₂, CH₄

## Tujuan Notebook

Notebook ini menggantikan cakupan wilayah dari **Kabupaten Lamongan** (notebook sebelumnya)
menjadi **Kecamatan Kedungpring** secara spesifik, lalu melanjutkan alur kerja hingga
tahap **ekstraksi fitur** menggunakan TSFEL:

1. **Ekstraksi data** — menarik ulang data Sentinel-5P L2 dengan AOI baru: Kecamatan Kedungpring.
2. **Preprocessing sebelum ekstraksi fitur**:
   - Data dipersempit sesuai batas Kecamatan Kedungpring.
   - Deteksi outlier (Isolation Forest, seperti pada notebook analisis sebelumnya).
   - Imputasi missing value hingga **tidak ada NaN tersisa**.
3. **Ekstraksi fitur** menggunakan **TSFEL** (Time Series Feature Extraction Library) menjadi
   kurang lebih **65 fitur**, dikelompokkan ke dalam 3 domain:
   - **Statistical** (statistik deskriptif)
   - **Temporal** (pola berbasis waktu)
   - **Spectral** (pola frekuensi)

## Data Wilayah Kecamatan Kedungpring

Berdasarkan dokumen resmi *Laporan Kinerja Instansi Pemerintah (LKjIP) Kecamatan Kedungpring 2024*
dan Wikipedia:

| Atribut | Nilai |
| ------- | ----- |
| Luas wilayah | 84,54 km² |
| Jumlah desa | 23 desa |
| Batas Utara | Kecamatan Babat |
| Batas Timur | Kecamatan Sugio |
| Batas Selatan | Kecamatan Ngimbang |
| Batas Barat | Kecamatan Modo |
| Rentang koordinat | 112°10′01″–112°13′28″ BT, 07°08′19″–07°12′27″ LS |
| Titik pusat | 7°11′02″ LS, 112°12′15″ BT |

> **Catatan:** Bounding box di bawah dikonversi langsung dari koordinat resmi di atas
> (derajat-menit-detik → desimal). Bentuknya tetap berupa **persegi pembatas (bounding
> box)**, bukan poligon administratif presisi. Untuk poligon batas desa/kecamatan yang
> akurat, gunakan data resmi (mis. **Kemendagri**, **BIG**, atau **GADM**) yang dibaca via
> `geopandas`, lalu ganti `KEDUNGPRING_POLYGON` di bawah.

## 2. Import Pustaka

Selain pustaka yang sudah dipakai pada notebook sebelumnya (`openeo`, `pandas`, `xarray`,
`sklearn`), notebook ini menambahkan **`tsfel`** untuk ekstraksi fitur deret waktu.

In [1]:
import os
import openeo
import xarray as xr
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import tsfel

NC_DIR = "../data/nc/"
CSV_DIR = "../data/csv/"
FEATURE_DIR = "../data/features/"
os.makedirs(NC_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(FEATURE_DIR, exist_ok=True)

print("Folder output siap:")
for d in [NC_DIR, CSV_DIR, FEATURE_DIR]:
    print(" -", os.path.abspath(d))

Folder output siap:
 - d:\Semester 5\Proyek sain data\PSD\data\nc
 - d:\Semester 5\Proyek sain data\PSD\data\csv
 - d:\Semester 5\Proyek sain data\PSD\data\features


## 3. Area of Interest (AOI): Kecamatan Kedungpring

`KEDUNGPRING_BBOX` dipakai sebagai `spatial_extent` saat `load_collection`.
`KEDUNGPRING_POLYGON` dipakai sebagai geometry pada `aggregate_spatial`.

In [2]:
# Bounding box Kecamatan Kedungpring (dikonversi dari koordinat resmi LKjIP 2024)
KEDUNGPRING_BBOX = {
    "west": 112.1669,
    "south": -7.2075,
    "east": 112.2244,
    "north": -7.1386,
}

# Polygon sederhana (mengikuti bounding box resmi) — ganti dengan geometri administratif
# presisi (GADM/BIG/Kemendagri) bila tersedia.
KEDUNGPRING_POLYGON = {
    "type": "Polygon",
    "coordinates": [[
        [112.1669, -7.1386],
        [112.2244, -7.1386],
        [112.2244, -7.2075],
        [112.1669, -7.2075],
        [112.1669, -7.1386],
    ]]
}

KEDUNGPRING_CENTER = [-7.1839, 112.2042]  # [lat, lon]

print("AOI Kecamatan Kedungpring:", KEDUNGPRING_BBOX)

AOI Kecamatan Kedungpring: {'west': 112.1669, 'south': -7.2075, 'east': 112.2244, 'north': -7.1386}


## 4. Ekstraksi Data untuk AOI Kedungpring

Struktur fungsi ekstraksi sama seperti pada `1-ekstraksi-data.ipynb`, hanya `bbox` dan
`polygon` yang diganti menjadi wilayah Kecamatan Kedungpring. File output diberi akhiran
`_kedungpring` agar tidak tertimpa/tercampur dengan hasil level Kabupaten sebelumnya.

In [3]:
TEMPORAL_EXTENT = ["2025-08-24", "2026-08-24"]

POLLUTANTS = [
    {"band": "NO2", "label": "no2"},
    {"band": "CO",  "label": "co"},
    {"band": "SO2", "label": "so2"},
    {"band": "CH4", "label": "ch4"},
]

def extract_pollutant(connection, band, label, bbox, polygon, temporal_extent, output_dir, suffix):
    """Menarik satu polutan, agregasi harian (mean) + spasial (mean dalam polygon)."""
    datacube = connection.load_collection(
        "SENTINEL_5P_L2",
        spatial_extent=bbox,
        temporal_extent=temporal_extent,
        bands=[band],
    )
    daily_cube = datacube.aggregate_temporal_period(period="day", reducer="mean")
    spatial_result = daily_cube.aggregate_spatial(geometries=polygon, reducer="mean")

    output_path = os.path.join(output_dir, f"{label}_{suffix}.nc")
    print(f"[{label.upper()}] Mengunduh -> {output_path} ...")
    spatial_result.download(output_path, format="netCDF")
    print(f"[{label.upper()}] Selesai.\n")
    return output_path


def nc_to_csv(nc_path, label, csv_dir, suffix):
    ds = xr.open_dataset(nc_path)
    df = ds.to_dataframe().reset_index()
    rename_map = {c: "date" for c in df.columns if "time" in c.lower() or "date" in c.lower()}
    df = df.rename(columns=rename_map)
    csv_path = os.path.join(csv_dir, f"{label}_{suffix}.csv")
    df.to_csv(csv_path, index=False)
    print(f"[{label.upper()}] CSV disimpan -> {csv_path} ({len(df)} baris)")
    return csv_path

In [4]:
# Koneksi & otentikasi (device code flow) — sama seperti notebook 1
connection = openeo.connect("openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()
print("Terhubung sebagai:", connection.describe_account())

Authenticated using refresh token.
Terhubung sebagai: {'info': {'oidc_userinfo': {'email': 'triswanti1395@gmail.com', 'email_verified': True, 'family_name': "Jannatul Ma'wa", 'given_name': 'Triswanti', 'name': "Triswanti Jannatul Ma'wa", 'preferred_username': 'triswanti1395@gmail.com', 'sub': '08868660-f1d0-4cae-bac7-ad9a53fa7209'}}, 'name': "Triswanti Jannatul Ma'wa", 'user_id': '08868660-f1d0-4cae-bac7-ad9a53fa7209'}


In [5]:
nc_paths = {}
csv_paths = {}

for p in POLLUTANTS:
    try:
        nc_path = extract_pollutant(
            connection=connection,
            band=p["band"],
            label=p["label"],
            bbox=KEDUNGPRING_BBOX,
            polygon=KEDUNGPRING_POLYGON,
            temporal_extent=TEMPORAL_EXTENT,
            output_dir=NC_DIR,
            suffix="kedungpring",
        )
        nc_paths[p["label"]] = nc_path
        csv_paths[p["label"]] = nc_to_csv(nc_path, p["label"], CSV_DIR, suffix="kedungpring")
    except Exception as e:
        print(f"[{p['label'].upper()}] Gagal diproses: {e}\n")

print("File CSV yang berhasil dibuat:")
for label, path in csv_paths.items():
    print(f" - {label}: {path}")

[NO2] Mengunduh -> ../data/nc/no2_kedungpring.nc ...
[NO2] Gagal diproses: [Errno 13] Permission denied

[CO] Mengunduh -> ../data/nc/co_kedungpring.nc ...
[CO] Selesai.

[CO] CSV disimpan -> ../data/csv/co_kedungpring.csv (202 baris)
[SO2] Mengunduh -> ../data/nc/so2_kedungpring.nc ...
[SO2] Selesai.

[SO2] CSV disimpan -> ../data/csv/so2_kedungpring.csv (219 baris)
[CH4] Mengunduh -> ../data/nc/ch4_kedungpring.nc ...
[CH4] Selesai.

[CH4] CSV disimpan -> ../data/csv/ch4_kedungpring.csv (5 baris)
File CSV yang berhasil dibuat:
 - co: ../data/csv/co_kedungpring.csv
 - so2: ../data/csv/so2_kedungpring.csv
 - ch4: ../data/csv/ch4_kedungpring.csv


## 5. Preprocessing Sebelum Ekstraksi Fitur

Tiga langkah preprocessing dilakukan berurutan pada data Kecamatan Kedungpring:

1. **Data sudah dipersempit sesuai kecamatan** — dilakukan pada tahap ekstraksi (Bagian 4)
   melalui `KEDUNGPRING_BBOX`/`KEDUNGPRING_POLYGON`, bukan lagi cakupan seluruh Kabupaten.
2. **Deteksi outlier** dengan `IsolationForest(contamination=0.05)`, mengikuti pendekatan
   yang sama seperti pada notebook analisis sebelumnya.
3. **Imputasi missing value hingga terisi semua** — interpolasi berbasis waktu, dilanjutkan
   `ffill`/`bfill` untuk memastikan **tidak ada satupun NaN tersisa** sebelum masuk ke
   tahap ekstraksi fitur (TSFEL tidak dapat memproses data dengan NaN).

In [6]:
# 5.1 Memuat & menggabungkan seluruh polutan menjadi satu dataframe
df_list = []
for pollutant, path in csv_paths.items():
    temp_df = pd.read_csv(path)
    date_col = [c for c in temp_df.columns if "date" in c.lower()][0]
    value_col = [c for c in temp_df.columns if c != date_col][-1]
    temp_df = temp_df[[date_col, value_col]].rename(columns={date_col: "date", value_col: pollutant})
    temp_df["date"] = pd.to_datetime(temp_df["date"])
    df_list.append(temp_df)

    df = df_list[0]
for other in df_list[1:]:
    df = df.merge(other, on="date", how="outer")

    df = df.sort_values("date").reset_index(drop=True)
    pollutant_cols = [c for c in df.columns if c != "date"]

    print(f"Data gabungan Kedungpring: {df.shape[0]} baris, kolom polutan: {pollutant_cols}")
    df.head()

IndexError: list index out of range

In [ ]:
# 5.2 Deteksi outlier per polutan (Isolation Forest, contamination=0.05)
df_outlier_flags = df.copy()

for pollutant in pollutant_cols:
    valid_mask = df_outlier_flags[pollutant].notna()
    flags = pd.Series(False, index=df_outlier_flags.index)

    if valid_mask.sum() > 0:
        model = IsolationForest(contamination=0.05, random_state=42)
        preds = model.fit_predict(df_outlier_flags.loc[valid_mask, [pollutant]])
        flags.loc[valid_mask] = (preds == -1)

    df_outlier_flags[f"{pollutant}_outlier"] = flags
    print(f"{pollutant.upper():<5}: {flags.sum()} outlier terdeteksi dari {valid_mask.sum()} data valid")

df_outlier_flags.head()

NameError: name 'df' is not defined

In [ ]:
# 5.3 Imputasi missing value hingga TIDAK ADA NaN tersisa
df_clean = df.set_index("date").copy()

print("Missing value SEBELUM imputasi:")
print(df_clean[pollutant_cols].isna().sum())

# Langkah 1: interpolasi linear berbasis waktu
df_clean[pollutant_cols] = df_clean[pollutant_cols].interpolate(method="time")

# Langkah 2: forward-fill lalu backward-fill untuk menutup sisa NaN di ujung deret waktu
df_clean[pollutant_cols] = df_clean[pollutant_cols].ffill().bfill()

# Langkah 3 (jaga-jaga): jika kolom polutan seluruhnya NaN (tidak ada data sama sekali),
# ffill/bfill tidak akan mengisi apa pun -> isi dengan 0 sebagai fallback terakhir
df_clean[pollutant_cols] = df_clean[pollutant_cols].fillna(0)

df_clean = df_clean.reset_index()

print("\nMissing value SETELAH imputasi:")
print(df_clean[pollutant_cols].isna().sum())

assert df_clean[pollutant_cols].isna().sum().sum() == 0, "Masih ada NaN tersisa!"
print("\nSemua missing value berhasil diisi (0 NaN tersisa).")

df_clean.head()

## 6. Ekstraksi Fitur dengan TSFEL

**TSFEL (Time Series Feature Extraction Library)** mengekstrak fitur deret waktu ke dalam
tiga domain:

| Domain | Contoh Fitur | Fokus |
| ------ | ------------ | ----- |
| **Statistical** | mean, median, std, skewness, kurtosis, kuartil | Distribusi nilai |
| **Temporal** | autocorrelation, slope, zero crossing rate | Pola sepanjang waktu |
| **Spectral** | FFT mean/std, spectral entropy, power bandwidth | Pola frekuensi |

Ekstraksi dilakukan **per domain secara terpisah** (`tsfel.get_features_by_domain(domain)`),
sehingga hasilnya otomatis sudah terkelompok berdasarkan domain — tidak perlu
pengelompokan manual tambahan.

Karena deret waktu di sini adalah satu rangkaian nilai harian per polutan (bukan sinyal
frekuensi tinggi), seluruh data diperlakukan sebagai **satu window utuh**
(`window_size=None`), sehingga setiap polutan menghasilkan **satu baris fitur** yang
merepresentasikan keseluruhan periode observasi.

In [ ]:
def extract_tsfel_features(signal_df, fs=1):
    """
    Mengekstrak fitur TSFEL untuk setiap kolom polutan pada signal_df,
    dikelompokkan ke dalam domain statistical, temporal, dan spectral.

    Parameters
    ----------
    signal_df : pd.DataFrame
        Dataframe berisi kolom-kolom polutan (tanpa NaN).
    fs : int
        Sampling frequency (1 = harian, karena agregasi data sudah per hari).

    Returns
    -------
    dict[str, pd.DataFrame]
        Dictionary berisi dataframe fitur per domain: 'statistical', 'temporal', 'spectral'.
    pd.DataFrame
        Dataframe gabungan seluruh domain (untuk disimpan sebagai satu file).
    """
    domains = ["statistical", "temporal", "spectral"]
    domain_results = {}

    for domain in domains:
        cfg = tsfel.get_features_by_domain(domain)
        feats = tsfel.time_series_features_extractor(
            cfg, signal_df, fs=fs, verbose=0, window_size=len(signal_df)
        )
        domain_results[domain] = feats

    combined = pd.concat(domain_results.values(), axis=1)
    return domain_results, combined


feature_input = df_clean.set_index("date")[pollutant_cols]

domain_features, all_features = extract_tsfel_features(feature_input, fs=1)

for domain, feats in domain_features.items():
    print(f"Domain {domain.capitalize():<12}: {feats.shape[1]} fitur")

print(f"\nTotal fitur gabungan: {all_features.shape[1]} fitur")
print(f"(Target referensi TSFEL: ±65 fitur per sinyal — jumlah aktual tergantung versi "
      f"pustaka TSFEL yang terpasang.)")

In [ ]:
# Pratinjau fitur per domain
print("Contoh fitur domain STATISTICAL:")
display(domain_features["statistical"].head())

print("\nContoh fitur domain TEMPORAL:")
display(domain_features["temporal"].head())

print("\nContoh fitur domain SPECTRAL:")
display(domain_features["spectral"].head())

In [ ]:
# Menyimpan hasil fitur ke CSV, terpisah per domain + gabungan
for domain, feats in domain_features.items():
    out_path = os.path.join(FEATURE_DIR, f"fitur_{domain}_kedungpring.csv")
    feats.to_csv(out_path, index=False)
    print(f"Tersimpan: {out_path}")

combined_path = os.path.join(FEATURE_DIR, "fitur_gabungan_kedungpring.csv")
all_features.to_csv(combined_path, index=False)
print(f"Tersimpan: {combined_path}")

## 7. Ringkasan

| Tahap | Hasil |
| ----- | ----- |
| AOI | Kecamatan Kedungpring, Kabupaten Lamongan (bukan lagi seluruh Kabupaten) |
| Missing value | Diimputasi hingga 0 NaN tersisa (interpolasi waktu + ffill/bfill) |
| Outlier | Ditandai per polutan menggunakan Isolation Forest (`contamination=0.05`) |
| Fitur | Diekstrak dengan TSFEL, dikelompokkan domain **statistical**, **temporal**, **spectral** |
| Output | `../data/features/fitur_statistical_kedungpring.csv`, `fitur_temporal_kedungpring.csv`, `fitur_spectral_kedungpring.csv`, `fitur_gabungan_kedungpring.csv` |

Hasil fitur ini siap dipakai untuk tahap selanjutnya (mis. pemodelan/klasifikasi kualitas
udara), dan file-file CSV di atas dapat direferensikan pada chapter analisis berikutnya
di Jupyter Book.